# 05 · Modules - hub anti-correlation (Gate 3)

On the **Mixed hub**, correlate every **Gast OR pSom** gene against both module
scores and measure the correlation between the two correlation vectors
(Both-module genes excluded from r). A negative r confirms the two embryonic
programs are mutually exclusive within the hub - the bifurcation signature
(**Gate 3**; reference anchor r ~ -0.49). Deliverable: `modules.pdf`.

In [1]:
import os, sys, glob
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import numpy as np, pandas as pd
import scanpy as sc, scanpy.external as sce, scFates as scf
import matplotlib.pyplot as plt
sc.settings.verbosity = 1

In [2]:
from scipy.stats import pearsonr
from adjustText import adjust_text

In [3]:
# cwd = <root>/mesencontext/claude/<Category>/<Dataset>
name = os.path.basename(os.getcwd())                    # dataset (folder minus 'Data_')
cat  = os.path.basename(os.path.dirname(os.getcwd()))   # Category partition
pth  = os.path.join(*([os.pardir]*4))                   # -> EMToolbox root
pth_data = os.path.join(pth, 'data')
sys.path.append(os.path.join(pth, 'mesencontext'))
from scripts.load import load_CCCA_adata

pth_in  = os.path.join(pth_data, 'unzip', 'CCCA', 'Data_' + name)        # raw input
pth_out = os.path.join(pth_data, 'cancer', 'vertebrate', 'CCCA', 'claude', cat, name)  # h5ads
pth_fig = os.path.join(pth, 'figures', 'mesencontext', 'claude', cat, name)
pth_cb  = os.path.join(pth_fig, 'colorbars')
os.makedirs(pth_out, exist_ok=True); os.makedirs(pth_cb, exist_ok=True)

In [4]:
c_dict = {'Gast': plt.cm.tab20c(12), 'pSom': plt.cm.tab20c(8), 'Both': 'darkgrey'}
adata = sc.read_h5ad(os.path.join(pth_out, 'signatures.h5ad'))
feat = os.path.join(pth_data, 'features', 'biomart')
gast = pd.read_csv(os.path.join(feat,'GSE162534.csv')).hsapiens
psom = pd.read_csv(os.path.join(feat,'GSE229103.csv')).hsapiens
g = pd.merge(gast, psom, 'outer').hsapiens.drop_duplicates()
g = g.loc[g.isin(adata.var_names)]
hub = adata[adata.obs.celltype=='Mixed', g].copy()

### Per-gene module correlations -> rank

In [5]:
X = pd.DataFrame(hub.X.toarray(), index=hub.obs_names, columns=g)
X = X.loc[:, X.nunique(0) > 1]
R = pd.DataFrame(index=X.columns)
R['Gast'] = pearsonr(X, hub.obs['GSE162534'].values[:,None], axis=0).statistic
R['pSom'] = pearsonr(X, hub.obs['GSE229103'].values[:,None], axis=0).statistic
rank = R.abs().sum(1).rank(ascending=False)
R = R.loc[rank.sort_values().index]
gs, ps = set(gast), set(psom); ix = R.index
msk_dict = {'Gast': ix.isin(gs) & ~ix.isin(ps),
            'pSom': ~ix.isin(gs) & ix.isin(ps),
            'Both': ix.isin(gs) & ix.isin(ps)}
for mod, m in msk_dict.items():
    R.loc[m, 'Module'] = mod
R['Module'] = pd.Categorical(R.Module, list(reversed(list(msk_dict))), ordered=True)
rval = round(float(np.corrcoef(*R.loc[~msk_dict['Both'], ['Gast','pSom']].values.T)[0,1]), 2)
print('HUB module correlation r =', rval, ' (reference ~ -0.49)')
print('Gate 3:', 'PASS' if rval < -0.2 else 'REVIEW', '| hub n =', hub.n_obs, '| genes =', X.shape[1])

HUB module correlation r = -0.47  (reference ~ -0.49)
Gate 3: PASS | hub n = 466 | genes = 139


### Gate 3 - Gast/pSom mutual-exclusivity scatter (deliverable)

In [6]:
fig, ax = plt.subplots(1, 1, figsize=(3.5,3.5))
for mod, sdf in R.groupby('Module', observed=True):
    sdf.plot(kind='scatter', x='Gast', y='pSom', color=c_dict[mod], label=mod,
             s=25, alpha=.6, linewidth=0, ax=ax)
ax.axvline(0, color='k', lw=1, ls='--', zorder=0)
ax.axhline(0, color='k', lw=1, ls='--', zorder=0)
xl = abs(np.asarray(ax.get_xlim())).max(); yl = abs(np.asarray(ax.get_ylim())).max()
ax.set_xlim(-xl, xl); ax.set_ylim(-yl, yl)
texts = [ax.text(R.loc[i].Gast, R.loc[i].pSom, i, fontsize=6.25, fontstyle='italic')
         for i in R.head(20).index]
adjust_text(texts=texts)
ax.legend(title='Module', loc='lower left', title_fontsize=7.5, handlelength=1,
          handletextpad=.5, borderaxespad=.65, fontsize=7.5, reverse=True)
ax.text(.99, .01, r'$r=$'+f'{rval}', transform=ax.transAxes, fontsize=10,
        color='tab:red', ha='right', va='bottom')
ax.set_title('Mixed Population')
ax.set_xlabel(r'Gast Correlation ($r$)'); ax.set_ylabel(r'pSom Correlation ($r$)')
fig.savefig(os.path.join(pth_fig,'modules.pdf'), bbox_inches='tight')